# MMLU baseline for the original model

This notebook measures MMLU performance of the loaded model **before any editing**. It calls `evaluate_model_on_mmlu(...)` directly, so it does not reuse the cached baseline from the experiment pipeline.

In [1]:
import os 

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import json
import sys
from pathlib import Path

import pandas as pd

from heretic.config import Settings
from heretic.model import Model
from evaluate.mmlu import evaluate_model_on_mmlu

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

# Change these values when you want to compare modes.
MMLU_CONFIG = {
    "enabled": True,
    "dataset": "cais/mmlu",
    "subset": "all",
    "split": "test",
    "mode": "zero_shot",
    "answer_mode": "generate",  # "generate" or "logits"
    "n_shots": 0,
    "sample_size": 100,
    "sample_seed": 42,
    "max_new_tokens": 2048,  # room for `...` then final A–D
    "store_predictions": True,
}

original_argv = sys.argv.copy()
try:
    sys.argv = [sys.argv[0]] if sys.argv else ["notebook"]
    settings = Settings(
        model=MODEL_NAME,
        batch_size=16,
        max_response_length=4096,
        system_prompt="You are a helpful assistant.",
    )
finally:
    sys.argv = original_argv

model = Model(settings)

# Direct call: evaluates the currently loaded model as-is, without any editing.
result = evaluate_model_on_mmlu(model, MMLU_CONFIG)
summary = result["summary"]

print("MMLU summary for the original model")
print(json.dumps(summary, indent=2, ensure_ascii=False))

predictions = result.get("predictions", result.get("prediction_preview", []))
preview_rows = []
for row in predictions[:20]:
    preview_rows.append(
        {
            "subject": row.get("subject"),
            "question": row.get("question"),
            "correct_letter": row.get("correct_letter"),
            "predicted_letter": row.get("predicted_letter"),
            "is_correct": row.get("is_correct"),
            "raw_response": row.get("raw_response"),
            "choice_scores": row.get("choice_scores"),
        }
    )

preview_df = pd.DataFrame(preview_rows)
display(preview_df)

output_dir = Path("results") / "notebooks"
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "mmlu_original_model_baseline.json"
output_file.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved full baseline result to: {output_file}")


/home/rinya/miniconda3/envs/heretic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model deepseek-ai/DeepSeek-R1-Distill-Qwen-7B...

* Trying dtype auto...

Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.15s/it]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Ok

...

* attn.o_proj: 1 matrices per layer

* mlp.down_proj: 1 matrices per layer

MMLU summary for the original model
{
  "accuracy": 0.54,
  "correct": 54,
  "total": 100,
  "invalid_predictions": 0,
  "by_subject": {
    "anatomy": {
      "accuracy": 0.0,
      "correct": 0,
      "total": 1,
      "invalid_predictions": 0
    },
    "business_ethics": {
      "accuracy": 1.0,
      "correct": 2,
      "total": 2,
      "invalid_predictions": 0
    },
    "clinical_knowledge": {
      "accuracy": 0.25,
      "correct": 1,
      "total": 4,
      "invalid_predictions": 0
    },
    "college_mathematics": {
      "accuracy": 0.0,
      "correct": 0,
      "total": 2,
      "invalid_predictions": 0
    },
    "college_medicine": {
      "accuracy": 0.5,
      "correct": 1,
      "total": 2,
      "invalid_predictions": 0
    },
    "college_physics": {
      "accuracy": 0.0,
      "correct": 0,
      "total": 1,
      "invalid_predictions": 0
    },
    "computer_security": {
      "accuracy": 1.0,
      "correct": 2,
      "total": 2,
      "invalid_predictions": 0

,subject,question,correct_letter,predicted_letter,is_correct,raw_response,choice_scores
0,anatomy,Which of the following anatomical regions of a...,A,B,False,"Okay, so I have this question about the anatom...",None
1,business_ethics,_______ theory can be described as a code of c...,C,C,True,"Okay, so I have this multiple-choice question ...",None
2,business_ethics,What term can be used to describe 'the hypothe...,A,A,True,"Okay, so I have this multiple-choice question ...",None
3,clinical_knowledge,The key attribute in successful marathon runni...,D,D,True,"Okay, so I have this question about successful...",None
4,clinical_knowledge,Which of the following is true of calcium meta...,B,A,False,"Okay, so I have this question about calcium me...",None
5,clinical_knowledge,Why should careful consideration be given to p...,B,A,False,"Okay, so I have this multiple-choice question ...",None
6,clinical_knowledge,Phosphocreatine in the muscle cell is found in:,D,C,False,"Okay, so I have this question about phosphocre...",None
7,college_mathematics,Suppose A and B are n × n matrices with real e...,A,C,False,"Okay, so I have this question about matrices a...",None
8,college_mathematics,Suppose A is a 3 × 3 matrix such that det(A − ...,D,A,False,"Okay, so I have this problem about a 3x3 matri...",None
9,college_medicine,Triacylglycerides consist of I. A ribose backb...,D,D,True,"Okay, so I have this question about triacylgly...",None


Saved full baseline result to: results/notebooks/mmlu_original_model_baseline.json
